# Brainana Lite — Volumetric T1w Pipeline

Single-subject T1w volumetric preprocessing (no surface reconstruction, no Nextflow).

**Pipeline:** BIDS naming → synthesis → conform → skull strip/segmentation → bias correction → registration → atlas backprojection

**Local:** run `scripts/setup_brainana_lite.sh` once, then set `RUN_LOCAL = True` and paths in Cell 1.

**Colab:** set `RUN_LOCAL = False`; Cell 2 installs everything from GitHub automatically.


In [ ]:
from pathlib import Path

# ═══════════════════════════════════════════════════════════════
# USER SETTINGS
# ═══════════════════════════════════════════════════════════════
RUN_LOCAL        = True
BRAINANA_VERSION = "v1.0.0"

# Local paths (when RUN_LOCAL = True)
BRAINANA_ENV_DIR = str(Path.cwd() / "brainana_lite_env")
INPUT_DIR        = str(Path.cwd() / "t1w_input")
OUTPUT_DIR       = str(Path.cwd() / "brainana_lite_output")

# Colab paths (when RUN_LOCAL = False)
DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks/BrainanaLite"

# Pipeline settings
SUBJECT_ID = "sub01"   # without "sub-" prefix
SESSION_ID = None      # e.g. "01" or None
TEMPLATE   = "NMT2Sym" # NMT2Sym | NMT2Asym | MEBRAINS | Yerkes19 | D99
USE_GPU    = True


In [ ]:
import os
import re
import shutil
import subprocess
import sys
import logging
from pathlib import Path

# ── Resolve paths by mode ────────────────────────────────────────────────────
if RUN_LOCAL:
    WORK_DIR = Path.cwd()
    BRAINANA_DIR = Path(BRAINANA_ENV_DIR) / "brainana"
    if not BRAINANA_DIR.exists():
        raise FileNotFoundError(
            f"Brainana not found at {BRAINANA_DIR}. Run scripts/setup_brainana_lite.sh first."
        )
    INPUT_DIR = Path(INPUT_DIR)
    OUTPUT_DIR = Path(OUTPUT_DIR)
    env_sh = Path(BRAINANA_ENV_DIR) / "env.sh"
    if env_sh.exists():
        for line in env_sh.read_text().splitlines():
            if line.startswith("export PATH="):
                path_val = line.split("=", 1)[1].strip().strip('"').replace("${PATH}", os.environ.get("PATH", ""))
                os.environ["PATH"] = path_val
                print(f"Loaded PATH from {env_sh}")
                break
    print(f"Running locally from: {WORK_DIR}")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path(DRIVE_DIR)
    BRAINANA_DIR = Path("/content/brainana")
    INPUT_DIR = WORK_DIR / "t1w_input"
    OUTPUT_DIR = WORK_DIR / "brainana_lite_output"
    print(f"Running on Colab from: {WORK_DIR}")

    # Colab inline setup (equivalent to setup_brainana_lite.sh)
    if not BRAINANA_DIR.exists():
        subprocess.run([
            "git", "clone", "--branch", BRAINANA_VERSION, "--depth", "1",
            "https://github.com/xingyu-liu/brainana.git", str(BRAINANA_DIR)
        ], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(BRAINANA_DIR)], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "antspyx", "nibabel", "nilearn", "matplotlib", "torch", "torchvision",
        "pyyaml", "yacs", "h5py", "pandas", "scipy", "scikit-image", "scikit-learn",
        "Pillow", "pybids", "packaging", "psutil", "requests", "torchio", "tqdm", "seaborn"
    ], check=True)

    # FSL + AFNI via neurodebian
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "fsl", "afni"], check=True)
    fsl_dir = Path("/usr/share/fsl")
    os.environ["FSLDIR"] = str(fsl_dir)
    os.environ["PATH"] = f"{fsl_dir}/bin:/usr/lib/afni/bin:" + os.environ.get("PATH", "")

    import torch
    if torch.cuda.is_available():
        fireants_dir = Path("/content/fireants")
        if not fireants_dir.exists():
            subprocess.run(["git", "clone", "--quiet", "https://github.com/rohitrango/fireants", str(fireants_dir)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(fireants_dir)], check=True)
        fused = fireants_dir / "fused_ops"
        if fused.exists():
            subprocess.run([sys.executable, str(fused / "setup.py"), "build_ext", "install"], cwd=str(fused), check=False)
        print("FireANTs installed.")

# ── ANTs binaries from antspyx ───────────────────────────────────────────────
try:
    import ants
    ants_bin = Path(ants.get_ants_path()) / "bin"
    if ants_bin.is_dir():
        os.environ["PATH"] = f"{ants_bin}:{os.environ.get('PATH', '')}"
        print(f"ANTs on PATH: {ants_bin}")
except Exception as exc:
    print(f"WARNING: could not add antspyx ANTs to PATH: {exc}")

# ── Verify required system binaries ───────────────────────────────────────────
REQUIRED_BINS = ["flirt", "fslmaths", "fslstats", "3dresample", "antsRegistration", "N4BiasFieldCorrection"]
missing = [b for b in REQUIRED_BINS if shutil.which(b) is None]
if missing:
    raise RuntimeError(
        "Missing required binaries on PATH: " + ", ".join(missing) +
        "\nLocal: install FSL + AFNI, run setup_brainana_lite.sh, then source env.sh"
    )
print("All required binaries found.")

# ── Patch QC module (surface recon import optional for lite) ─────────────────
snapshots_py = BRAINANA_DIR / "src/nhp_mri_prep/quality_control/snapshots.py"
if snapshots_py.exists():
    text = snapshots_py.read_text()
    old = "from fastsurfer_surfrecon.io.surface import convert_fs_surface_to_gifti"
    new = (
        "try:\n    from fastsurfer_surfrecon.io.surface import convert_fs_surface_to_gifti\n"
        "except Exception:\n    convert_fs_surface_to_gifti = None"
    )
    if old in text and "convert_fs_surface_to_gifti = None" not in text:
        snapshots_py.write_text(text.replace(old, new))
        print("Patched snapshots.py for lite QC imports.")

# ── Shared imports ───────────────────────────────────────────────────────────
from IPython.display import Image, display

from nhp_mri_prep.config import load_config
from nhp_mri_prep.steps.types import StepInput
from nhp_mri_prep.steps.anatomical import (
    anat_synthesis,
    anat_conform,
    anat_skullstripping,
    anat_bias_correction,
    anat_registration,
    anat_backproject_atlases,
    anat_reproject_atlases_to_scanner,
)
from nhp_mri_prep.operations.registration import ants_apply_transforms
from nhp_mri_prep.utils.templates import resolve_template
from nhp_mri_prep.utils.bids import (
    create_bids_output_filename,
    create_synthesized_bids_filename,
    get_filename_stem,
    replace_bids_space,
)
from nhp_mri_prep.utils.nextflow import create_output_link, detect_modality
from nhp_mri_prep.quality_control import (
    create_conform_qc,
    create_skullstripping_qc,
    create_bias_correction_qc,
    create_registration_qc,
    create_atlas_segmentation_qc,
)

qc_logger = logging.getLogger("brainana_lite_qc")
if not qc_logger.handlers:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

# ── BIDS / output layout ─────────────────────────────────────────────────────
BIDS_PREFIX = f"sub-{SUBJECT_ID}"
if SESSION_ID:
    BIDS_PREFIX += f"_ses-{SESSION_ID}"

def subject_deriv_dir() -> Path:
    base = OUTPUT_DIR / "derivatives" / f"sub-{SUBJECT_ID}"
    return base / f"ses-{SESSION_ID}" / "anat" if SESSION_ID else base / "anat"

def subject_fig_dir() -> Path:
    base = OUTPUT_DIR / "derivatives" / f"sub-{SUBJECT_ID}"
    return base / f"ses-{SESSION_ID}" / "figures" if SESSION_ID else base / "figures"

ANAT_DIR = subject_deriv_dir()
FIG_DIR = subject_fig_dir()
WORK_ROOT = OUTPUT_DIR / "work"
ANAT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print(f"INPUT_DIR:  {INPUT_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"ANAT_DIR:   {ANAT_DIR}")
print(f"FIG_DIR:    {FIG_DIR}")


In [ ]:
# Step 0 — Convert raw NIfTI inputs to BIDS layout
INPUT_DIR.mkdir(parents=True, exist_ok=True)

candidates = sorted(
    p for p in INPUT_DIR.iterdir()
    if p.is_file() and (p.name.endswith(".nii.gz") or p.name.endswith(".nii"))
)
if not candidates:
    raise FileNotFoundError(f"No NIfTI files found in {INPUT_DIR}")

bids_anat_dir = OUTPUT_DIR / "bids_input" / f"sub-{SUBJECT_ID}"
if SESSION_ID:
    bids_anat_dir = bids_anat_dir / f"ses-{SESSION_ID}"
bids_anat_dir = bids_anat_dir / "anat"
bids_anat_dir.mkdir(parents=True, exist_ok=True)

bids_files = []
print("BIDS input mapping:")
for idx, src in enumerate(candidates, start=1):
    run_tag = f"_run-{idx:02d}" if len(candidates) > 1 else ""
    ses_tag = f"_ses-{SESSION_ID}" if SESSION_ID else ""
    dst_name = f"sub-{SUBJECT_ID}{ses_tag}{run_tag}_T1w.nii.gz"
    dst = bids_anat_dir / dst_name
    if src.resolve() != dst.resolve():
        shutil.copy2(src, dst)
    bids_files.append(dst)
    print(f"  {src.name} -> {dst.relative_to(OUTPUT_DIR)}")

bids_name = bids_files[0]
meta = {
    "subject_id": SUBJECT_ID,
    "session_id": SESSION_ID or "",
    "session_count": 1,
}


In [ ]:
# Config, templates, helpers
config = load_config()
config["anat"]["surface_reconstruction"]["enabled"] = False
config["template"]["output_space"] = f"{TEMPLATE}:res-05"
config["general"]["gpu_device"] = "auto" if USE_GPU else -1
config["general"]["anat_only"] = True

effective_output_space = config["template"]["output_space"]
template_file = Path(resolve_template(effective_output_space))
template_reg = Path(resolve_template(effective_output_space))

modality = "T1w"
bids_stem = get_filename_stem(bids_name)
bids_prefix_wo_modality = bids_stem.replace(f"_{modality}", "")


def publish_copy(src: Path, filename: str) -> Path:
    dst = ANAT_DIR / filename
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return dst


def show_qc(png_path: Path, title: str = "") -> None:
    if title:
        print(title)
    if png_path.exists():
        display(Image(filename=str(png_path)))
    else:
        print(f"QC image not found: {png_path}")

print(f"Template conform: {template_file}")
print(f"Template reg:     {template_reg}")


In [ ]:
# Step 1 — Synthesis (multi-run T1w within session)
synth_dir = WORK_ROOT / "synthesis"
synth_dir.mkdir(parents=True, exist_ok=True)

synth_out = anat_synthesis(anat_files=bids_files, working_dir=synth_dir, config=config)
synthesized = synth_out.metadata.get("synthesized", False)
is_subject_level = SESSION_ID is None

bids_output_filename, bids_name_downstream = create_synthesized_bids_filename(
    original_file=bids_name,
    modality=modality,
    is_subject_level=is_subject_level,
    synthesized=synthesized,
)

synth_file = synth_out.output_file
print(f"Synthesized: {synthesized}")
print(f"Working T1w: {synth_file}")
print(f"BIDS template: {bids_name_downstream}")

# bids_name used by downstream steps matches Nextflow downstream template
bids_name = Path(bids_name_downstream)
bids_stem = get_filename_stem(bids_name)
bids_prefix_wo_modality = bids_stem.replace(f"_{modality}", "")


In [ ]:
# Step 2 — Conform to template
conform_dir = WORK_ROOT / "conform"
conform_dir.mkdir(parents=True, exist_ok=True)

conform_out = anat_conform(
    StepInput(
        input_file=synth_file,
        working_dir=conform_dir,
        config=config,
        output_name="anat_conformed.nii.gz",
        metadata=meta,
    ),
    template_file=template_file,
)

conformed_path = publish_copy(
    conform_out.output_file,
    create_bids_output_filename(bids_name, suffix="desc-conform", modality=modality),
)

template_resampled = conform_out.additional_files.get("template_resampled", template_file)
conform_qc_png = FIG_DIR / f"{BIDS_PREFIX}_desc-conform_T1w.png"
create_conform_qc(
    conformed_file=str(conformed_path),
    template_file=str(template_resampled),
    save_f=str(conform_qc_png),
    modality="anat",
    logger=qc_logger,
)
show_qc(conform_qc_png, "Conform QC")
print(f"Conformed: {conformed_path}")


In [ ]:
# Step 3 — Skull stripping + segmentation
skull_dir = WORK_ROOT / "skullstrip"
skull_dir.mkdir(parents=True, exist_ok=True)

skull_out = anat_skullstripping(
    StepInput(
        input_file=conform_out.output_file,
        working_dir=skull_dir,
        config=config,
        output_name="anat_brain.nii.gz",
        metadata=meta,
    )
)

brain_mask_src = skull_out.additional_files["brain_mask"]
seg_src = skull_out.additional_files.get("segmentation")
atlas_name = skull_out.metadata.get("atlas_name", "ARM2")

mask_path = publish_copy(brain_mask_src, f"{bids_prefix_wo_modality}_space-T1w_desc-brain_mask.nii.gz")
brain_path = publish_copy(
    skull_out.output_file,
    f"{bids_prefix_wo_modality}_desc-skullstrip_{modality}_brain.nii.gz",
)
if seg_src:
    seg_path = publish_copy(
        seg_src,
        f"{bids_prefix_wo_modality}_space-T1w_desc-brain_atlas{atlas_name}.nii.gz",
    )

skull_qc_png = FIG_DIR / f"{BIDS_PREFIX}_desc-skullstrip_T1w.png"
create_skullstripping_qc(
    underlay_file=str(conformed_path),
    mask_file=str(mask_path),
    save_f=str(skull_qc_png),
    modality="anat",
    logger=qc_logger,
)
show_qc(skull_qc_png, "Skull stripping QC")
print(f"Brain mask: {mask_path}")


In [ ]:
# Step 4 — Bias correction
bias_dir = WORK_ROOT / "bias"
bias_dir.mkdir(parents=True, exist_ok=True)

bias_out = anat_bias_correction(
    StepInput(
        input_file=conform_out.output_file,
        working_dir=bias_dir,
        config=config,
        output_name="anat_bias_corrected.nii.gz",
        metadata=meta,
    ),
    brain_mask=brain_mask_src,
)

bias_head_path = publish_copy(
    bias_out.output_file,
    create_bids_output_filename(bids_name, suffix="desc-biascorrect", modality=modality),
)
if "brain" in bias_out.additional_files:
    bias_brain_src = bias_out.additional_files["brain"]
else:
    bias_brain_src = bias_out.output_file
bias_brain_path = publish_copy(
    bias_brain_src,
    f"{bids_prefix_wo_modality}_desc-biascorrect_{modality}_brain.nii.gz",
)

bias_qc_png = FIG_DIR / f"{BIDS_PREFIX}_desc-biascorrect_T1w.png"
create_bias_correction_qc(
    image_original=str(brain_path),
    image_corrected=str(bias_brain_path),
    save_f=str(bias_qc_png),
    modality="anat",
    logger=qc_logger,
)
show_qc(bias_qc_png, "Bias correction QC")

if seg_src:
    seg_qc_png = FIG_DIR / f"{BIDS_PREFIX}_desc-atlasSegmentation_T1w.png"
    create_atlas_segmentation_qc(
        underlay_file=str(bias_brain_path),
        atlas_file=str(seg_path),
        save_f=str(seg_qc_png),
        modality="anat",
        logger=qc_logger,
    )
    show_qc(seg_qc_png, "Atlas segmentation QC")

print(f"Bias-corrected head:  {bias_head_path}")
print(f"Bias-corrected brain: {bias_brain_path}")


In [ ]:
# Publish Phase 1 (desc-preproc naming)
preproc_head = publish_copy(
    bias_head_path,
    create_bids_output_filename(bids_name, suffix="space-T1w_desc-preproc", modality=modality),
)
preproc_brain = publish_copy(
    bias_brain_path,
    f"{bids_prefix_wo_modality}_space-T1w_desc-preproc_{modality}_brain.nii.gz",
)

print("Phase 1 published:")
print(f"  {preproc_head.name}")
print(f"  {preproc_brain.name}")


In [ ]:
# Step 5 — Registration (brain -> template, apply to full head)
reg_dir = WORK_ROOT / "registration"
reg_dir.mkdir(parents=True, exist_ok=True)

# Compute transform on bias-corrected brain (matches Nextflow ANAT_REGISTRATION)
reg_brain_out = anat_registration(
    StepInput(
        input_file=bias_brain_src,
        working_dir=reg_dir,
        config=config,
        output_name="anat_registered_brain.nii.gz",
        metadata=meta,
    ),
    template_file=template_reg,
    template_name=TEMPLATE,
)

forward_xfm = reg_brain_out.additional_files["forward_transform"]
inverse_xfm = reg_brain_out.additional_files["inverse_transform"]

interpolation = config.get("registration", {}).get("interpolation", "BSpline")
apply_result = ants_apply_transforms(
    movingf=str(bias_head_path),
    moving_type=0,
    interpolation=interpolation,
    outputf_name="anat_registered_head.nii.gz",
    fixedf=str(template_reg),
    working_dir=str(reg_dir / "apply_head"),
    transformf=[str(forward_xfm)],
    reff=str(template_reg),
    logger=qc_logger,
)

registered_head = Path(apply_result["imagef_registered"])
reg_head_path = publish_copy(
    registered_head,
    create_bids_output_filename(
        bids_name,
        suffix=f"space-{TEMPLATE}_desc-preproc",
        modality=modality,
    ),
)

reg_qc_png = FIG_DIR / f"{BIDS_PREFIX}_desc-anat2template_T1w.png"
create_registration_qc(
    image_file=str(reg_head_path),
    template_file=str(template_reg),
    save_f=str(reg_qc_png),
    modality="anat2template",
    logger=qc_logger,
)
show_qc(reg_qc_png, "Registration QC")
print(f"Registered head: {reg_head_path}")


In [ ]:
# Apply brain mask to template space
mask_reg_dir = WORK_ROOT / "mask_to_template"
mask_reg_dir.mkdir(parents=True, exist_ok=True)

mask_apply = ants_apply_transforms(
    movingf=str(mask_path),
    moving_type=0,
    interpolation="NearestNeighbor",
    outputf_name="mask_registered.nii.gz",
    fixedf=str(template_reg),
    working_dir=str(mask_reg_dir),
    transformf=[str(forward_xfm)],
    reff=str(template_reg),
    logger=qc_logger,
    generate_tmean=False,
)

mask_registered = Path(mask_apply["imagef_registered"])
mask_template_name = f"{replace_bids_space(bids_prefix_wo_modality, TEMPLATE)}_desc-brain_mask.nii.gz"
mask_template_path = publish_copy(mask_registered, mask_template_name)
print(f"Template-space mask: {mask_template_path}")


In [ ]:
# Step 6 — Atlas backproject to T1w space
atlas_t1w_dir = WORK_ROOT / "atlas_t1w"
atlas_t1w_dir.mkdir(parents=True, exist_ok=True)

bp_out = anat_backproject_atlases(
    inverse_xfm=inverse_xfm,
    t1w_reference=bias_head_path,
    bids_name=preproc_head.name,
    working_dir=atlas_t1w_dir,
    config=config,
    template_dir=None,
)

atlas_t1w_pub = ANAT_DIR / "atlas_space-T1w"
atlas_t1w_pub.mkdir(parents=True, exist_ok=True)
atlas_t1w_files = []
for atlas_name_key, atlas_path in bp_out.additional_files.items():
    dst = atlas_t1w_pub / atlas_path.name
    shutil.copy2(atlas_path, dst)
    atlas_t1w_files.append(dst)
    print(f"  T1w atlas: {dst.name}")

print(f"Backprojected {len(atlas_t1w_files)} atlases to T1w space")


In [ ]:
# Step 7 — Atlas to scanner space
atlas_scanner_dir = WORK_ROOT / "atlas_scanner"
atlas_scanner_dir.mkdir(parents=True, exist_ok=True)

conform_inverse = conform_out.additional_files.get("inverse_transform")
if conform_inverse is None:
    raise FileNotFoundError("Conform inverse transform not found")

scanner_out = anat_reproject_atlases_to_scanner(
    atlas_files=atlas_t1w_files,
    conform_inverse_xfm=conform_inverse,
    scanner_reference=synth_file,
    working_dir=atlas_scanner_dir,
)

atlas_scanner_pub = ANAT_DIR / "atlas_space-scanner"
atlas_scanner_pub.mkdir(parents=True, exist_ok=True)
for _, atlas_path in scanner_out.additional_files.items():
    dst = atlas_scanner_pub / Path(atlas_path).name
    shutil.copy2(atlas_path, dst)
    print(f"  Scanner atlas: {dst.name}")

print("Atlas reprojection to scanner space complete")


In [ ]:
# Output summary
print("\nDerivatives tree:")
deriv_root = OUTPUT_DIR / "derivatives"
for root, dirs, files in os.walk(deriv_root):
    dirs[:] = [d for d in dirs if not d.startswith("_")]
    level = Path(root).relative_to(deriv_root).parts
    indent = "  " * len(level)
    print(f"{indent}{Path(root).name}/")
    for fname in sorted(files):
        print(f"{indent}  {fname}")

print("\nDone.")
